# LLaDA-8B-Base: BBC memorization training

Run these cells **in order in a fresh Google Colab GPU runtime**. Select an **A100 80 GB with high host RAM**; this tested setup requires at least **140 GiB available CPU RAM**. The notebook prints the actual hardware and stops if resources are insufficient.

This is the agreed **full-parameter continuation**: BF16 model computation, FP32 accumulated gradients, CPU FP32 master weights and AdamW moments; 4096-token blocks, microbatch 1, accumulation 16, **3 epochs / 183 optimizer updates**. The model and dataset revisions are frozen. It uses even indices after filtering publication dates to March 2025 onwards; the pinned dataset contains March–June 2025. There are 5,322 selected articles (5,315 nonempty), 3,941,989 tokens and 963 blocks.

All large downloads and serialization happen **on Colab's local disk**, never on your Windows computer. Final files upload to **My Drive / LLaDA-ft / checkpoints / RUN_NAME** via resumable Drive API uploads. Each model is approximately **16.03 GB / 14.93 GiB**; three epoch snapshots total about **48.1 GB**, plus temporary rolling snapshots. The quota check requires approximately 66.3 GB free: at most four models in Drive simultaneously, plus 2 GiB headroom. This is the budget for this run only; other jobs uploading into the same Drive can consume its headroom.

Expect roughly **3 h 40 min of training computation on the measured machine**, plus diagnostics and substantial upload time. A full untrained-base upload verifies transport before any training. It is explicitly labeled `base-transport-preflight`. Do not mistake it for a trained checkpoint. The training cell shows live progress and keeps the notebook kernel doing real work; Colab can still disconnect or terminate it.

Epoch snapshots and rolling snapshots are **model-only**. They support evaluation or a separately defined new training run, **not exact optimizer/RNG resumption**. A new notebook run starts from the pinned base. This notebook refuses to overwrite an existing run. Only an older rolling snapshot belonging to this run is removed after its replacement has been verified; epoch snapshots are retained.

The complete methods document, source code, configuration, data checksums, environment and upload receipts are saved with the run. Diagnostic losses are monitoring measurements, not a memorization/extraction test.

## 1. Check the actual runtime before downloading anything

In [ ]:
import os, sys, json, shutil, subprocess
from pathlib import Path
from datetime import datetime, timezone
import torch

assert Path('/content').is_dir(), 'Run this notebook in Google Colab.'
assert torch.cuda.is_available(), 'Select a GPU runtime first.'
gpu = torch.cuda.get_device_properties(0)
meminfo = dict(line.split(':', 1) for line in Path('/proc/meminfo').read_text().splitlines())
available_gib = int(meminfo['MemAvailable'].split()[0]) * 1024 / 2**30
disk_gib = shutil.disk_usage('/content').free / 2**30
print(json.dumps({'gpu': gpu.name, 'vram_gib': gpu.total_memory / 2**30,
                  'available_host_ram_gib': available_gib, 'free_colab_disk_gib': disk_gib,
                  'torch': torch.__version__}, indent=2))
assert gpu.total_memory >= 79 * 2**30, 'This protocol requires an 80 GB GPU.'
assert torch.cuda.is_bf16_supported(), 'BF16 support is required.'
assert available_gib >= 140, 'Select a high-RAM runtime: need 140 GiB AVAILABLE host RAM.'
assert disk_gib >= 45, 'Need at least 45 GiB free local disk for base and checkpoint staging.'

## 2. Authorize Google Drive
Approve the authentication and mount prompts. Credentials stay in Colab; the notebook does not print them.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/LLaDA-ft')
for name in ['datasets', 'docs', 'logs', 'runs']:
    (ROOT / name).mkdir(parents=True, exist_ok=True)

## 3. Install the compatible training environment
An isolated environment inherits Colab’s existing PyTorch and CUDA installation. The constraints prevent replacing PyTorch. Other Python dependencies are installed only on Colab.

In [ ]:
VENV = Path('/content/llada-bench-venv')
if not (VENV / 'bin/python').exists():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV)], check=True)
PYTHON = str(VENV / 'bin/python')
Path('/content/llada-constraints.txt').write_text('torch==' + str(torch.__version__) + '\n')
requirements = ['transformers==4.46.3', 'peft==0.14.0', 'huggingface-hub==0.36.0',
                'tokenizers==0.20.3', 'datasets==4.8.5', 'accelerate==1.15.0',
                'safetensors==0.7.0', 'google-api-python-client', 'google-auth-httplib2', 'psutil']
subprocess.run([PYTHON, '-m', 'pip', 'install', '--disable-pip-version-check',
                '--constraint', '/content/llada-constraints.txt', *requirements], check=True)
subprocess.run([PYTHON, '-c', 'import torch, transformers, peft, datasets, accelerate, safetensors, psutil; '
                'import google.auth, googleapiclient.discovery; '
                'print("TRAINING_IMPORTS_OK", torch.__version__, transformers.__version__)'], check=True)
os.environ.update(HF_HOME='/content/hf-cache-local', HF_DATASETS_CACHE='/content/hf-cache-local/datasets',
                  HF_HUB_DISABLE_XET='1', TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')

## 4. Run identity and frozen configuration
A unique name is generated once per notebook kernel. Keep the scientific settings unchanged to reproduce this protocol. An interrupted run cannot be exactly resumed; use a new name only when you intend to restart from Base.

In [ ]:
if 'RUN_NAME' not in globals():
    RUN_NAME = 'bbc_fp32_colab_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
CONFIG = {'run_name': 'bbc_fp32_3ep_v3_api', 'seed': 42, 'epochs': 3, 'sequence_length': 4096, 'microbatch': 1, 'accumulation': 16, 'gradient_dtype': 'float32', 'model_dtype': 'bfloat16', 'master_weight_dtype': 'float32', 'optimizer_state_dtype': 'float32', 'optimizer': 'AdamW', 'optimizer_device': 'cpu', 'peak_learning_rate': 1e-05, 'final_learning_rate': 1e-06, 'warmup_fraction': 0.05, 'decay_fraction': 0.1, 'betas': [0.9, 0.999], 'epsilon': 1e-08, 'weight_decay': 0.1, 'weight_decay_parameter_groups': 'all trainable parameters, including norms and biases if present', 'gradient_clip_norm': 1.0, 'mask_token_id': 126336, 'minimum_mask_probability': 0.001, 'random_length_probability': 0.01, 'random_length_min': 1, 'tail_policy': 'Unpadded final short block; retain all source tokens', 'packing': 'Frozen source-order article content plus EOS, no BOS or chat template; shuffle packed blocks each epoch', 'checkpointing': 'whole_layer_nonreentrant', 'cpu_threads': 12, 'diagnostic_blocks_per_group': 8, 'diagnostic_mask_draws_per_block': 4, 'checkpoint_epochs': [1, 2, 3], 'checkpoint_format': 'BF16 safetensors: validate on local Colab disk, resumable Drive API upload, server checksums, COMPLETE marker last', 'save_optimizer_state': False, 'resume_supported': False, 'methods_filename': 'LLADA_BBC_METHOD_v3.md', 'checkpoint_backend': 'drive_api', 'checkpoint_every_updates': 20, 'verify_full_checkpoint_before_training': True, 'model_path_override': '/content/llada-base-model'}
CONFIG.update(run_name=RUN_NAME, methods_filename=RUN_NAME + '_METHODS.md')
CONFIG_PATH = Path('/content/llada_training_config.json')
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))
assert not (ROOT / 'runs' / RUN_NAME).exists(), 'Run already exists; do not overwrite or restart it.'
print('RUN_NAME:', RUN_NAME)
print(json.dumps(CONFIG, indent=2))

## 5. Write the standalone implementation
These cells contain the actual reviewed training and upload implementation. They write small source files under `/content`; you can inspect or edit them before training. No files from the Windows workspace are needed.

In [ ]:
%%writefile /content/benchmark_llada.py
"""Shared paths and environment; no benchmark is launched by this module."""
import os
from pathlib import Path
ROOT = Path('/content/drive/MyDrive/LLaDA-ft')
os.environ.update(HF_HOME='/content/hf-cache-local', HF_DATASETS_CACHE='/content/hf-cache-local/datasets',
                  HF_HUB_DISABLE_XET='1', TOKENIZERS_PARALLELISM='false', USE_TF='0', USE_FLAX='0')

In [ ]:
%%writefile /content/drive_checkpoint_upload.py
"""Resumable Drive API uploads, with server-side size and MD5 verification.

Upload session URLs stay in memory and are never printed or stored on Drive.
Existing named files are reused only when content checksums match.
"""
from pathlib import Path
import hashlib
import io
import json
import time

FOLDER='application/vnd.google-apps.folder'
FIELDS='id,name,size,md5Checksum,sha256Checksum,webViewLink,parents,appProperties'


def quoted(value):
    return str(value).replace('\\','\\\\').replace("'","\\'")


def hashes(path):
    md5=hashlib.md5()
    sha=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):
            md5.update(chunk);sha.update(chunk)
    return md5.hexdigest(),sha.hexdigest()


class DriveCheckpointStore:
    def __init__(self,service=None):
        if service is None:
            import google.auth
            from googleapiclient.discovery import build
            credentials,_=google.auth.default()
            service=build('drive','v3',credentials=credentials,cache_discovery=False)
        self.service=service

    def children(self,parent,name=None):
        query=f"'{quoted(parent)}' in parents and trashed=false"
        if name is not None:
            query+=f" and name='{quoted(name)}'"
        result=[]
        token=None
        while True:
            response=self.service.files().list(q=query,fields=f'nextPageToken,files({FIELDS},mimeType)',
                pageSize=1000,pageToken=token).execute(num_retries=5)
            result.extend(response.get('files',[]))
            token=response.get('nextPageToken')
            if not token:
                return result

    def folder(self,parent,name,create=True):
        matches=[f for f in self.children(parent,name) if f['mimeType']==FOLDER]
        if len(matches)>1:
            raise RuntimeError(f'Ambiguous Drive folder name: {name}')
        if matches:
            return matches[0]['id']
        if not create:
            return None
        return self.service.files().create(body=dict(name=name,mimeType=FOLDER,parents=[parent]),fields='id').execute(num_retries=5)['id']

    def ensure_path(self,parts):
        parent='root'
        for part in parts:
            parent=self.folder(parent,part)
        return parent

    def write_state(self,parent,name,data):
        """Overwrite only the small API-owned run-state mirror, not model artifacts."""
        from googleapiclient.http import MediaIoBaseUpload
        existing=self.children(parent,name)
        if len(existing)>1:
            raise RuntimeError('Ambiguous state mirror')
        media=MediaIoBaseUpload(io.BytesIO(json.dumps(data,indent=2).encode()),mimetype='application/json',resumable=False)
        if existing:
            if existing[0].get('appProperties',{}).get('codex_role')!='run_state':
                raise RuntimeError('Refusing to overwrite a state file not created by this uploader')
            return self.service.files().update(fileId=existing[0]['id'],media_body=media,fields=FIELDS).execute(num_retries=5)
        return self.service.files().create(body=dict(name=name,parents=[parent],appProperties={'codex_role':'run_state'}),
            media_body=media,fields=FIELDS).execute(num_retries=5)

    def delete_own_periodic(self,receipt):
        """Collect only a previous periodic folder whose children match our upload receipt."""
        folder=receipt['folder_id']
        expected={v['id'] for v in receipt['files'].values()}
        actual={v['id'] for v in self.children(folder)}
        if actual!=expected:
            raise RuntimeError('Periodic snapshot has unexpected children; automatic cleanup refused')
        for file_id in expected:
            self.service.files().delete(fileId=file_id).execute(num_retries=5)
        self.service.files().delete(fileId=folder).execute(num_retries=5)

    def upload(self,path,parent,progress=None):
        from googleapiclient.http import MediaFileUpload
        path=Path(path)
        size=path.stat().st_size
        md5,sha=hashes(path)
        existing=[f for f in self.children(parent,path.name) if f.get('mimeType')!=FOLDER]
        if len(existing)>1:
            raise RuntimeError(f'Ambiguous Drive file name: {path.name}')
        if existing:
            remote=existing[0]
            if int(remote.get('size',-1))==size and remote.get('md5Checksum')==md5:
                return dict(**remote,verified_md5=md5,local_sha256=sha,reused=True)
            raise RuntimeError(f'Refusing to overwrite different content: {path.name}')
        media=MediaFileUpload(str(path),mimetype='application/octet-stream',chunksize=64*1024*1024,resumable=True)
        request=self.service.files().create(body=dict(name=path.name,parents=[parent],appProperties={'sha256':sha}),
            media_body=media,fields=FIELDS)
        response=None
        try:
            while response is None:
                status,response=request.next_chunk(num_retries=8)
                if status is not None and progress:
                    progress(path.name,status.resumable_progress,size)
        except Exception as error:
            status=getattr(getattr(error,'resp',None),'status',None)
            raise RuntimeError(f'Upload failed for {path.name}: {type(error).__name__}, HTTP {status}; resumable URL withheld') from None
        remote=self.service.files().get(fileId=response['id'],fields=FIELDS).execute(num_retries=5)
        if int(remote.get('size',-1))!=size or remote.get('md5Checksum')!=md5:
            raise RuntimeError(f'Drive verification failed for {path.name}')
        if remote.get('sha256Checksum') and remote['sha256Checksum']!=sha:
            raise RuntimeError(f'Drive SHA256 mismatch for {path.name}')
        return dict(**remote,verified_md5=md5,local_sha256=sha,reused=False)

    def upload_directory(self,path,parent,progress=None):
        path=Path(path)
        if not (path/'COMPLETE.json').is_file():
            raise RuntimeError('Local checkpoint must be validated before upload')
        remote_files={}
        for p in sorted(path.iterdir()):
            if p.is_file() and p.name!='COMPLETE.json':
                remote_files[p.name]=self.upload(p,parent,progress)
        # Completion marker is uploaded only after every shard and sidecar is verified.
        remote_files['COMPLETE.json']=self.upload(path/'COMPLETE.json',parent,progress)
        return dict(folder_id=parent,folder_url=f'https://drive.google.com/drive/folders/{parent}',files=remote_files,
            verified=True,total_bytes=sum(int(v.get('size',0)) for v in remote_files.values()))


def inspect_project():
    store=DriveCheckpointStore()
    project=store.folder('root','LLaDA-ft',create=False)
    if not project:
        raise RuntimeError('Project folder not found via Drive API')
    print('PROJECT_FOLDER_ID',project)
    print('QUOTA',json.dumps(store.service.about().get(fields='storageQuota').execute()))
    def walk(parent,parts,depth):
        for item in store.children(parent):
            p=parts+[item['name']]
            if item['mimeType']==FOLDER:
                if depth:
                    walk(item['id'],p,depth-1)
            else:
                print('FILE',json.dumps(dict(path='/'.join(p),id=item['id'],size=item.get('size'),md5=item.get('md5Checksum'))))
                if item['name']=='status.json' and 'bbc_fp32_3ep_v1' in p:
                    raw=store.service.files().get_media(fileId=item['id']).execute(num_retries=5)
                    print('API_RUN_STATUS',raw.decode('utf-8'))
    for name in ['checkpoints','runs']:
        folder=store.folder(project,name,create=False)
        if folder:
            walk(folder,[name],3)
        else:
            print('ABSENT_FOLDER',name)


if __name__=='__main__':
    inspect_project()

In [ ]:
%%writefile /content/train_llada_bbc.py
"""Protocol v1: full LLaDA BBC continuation with FP32 CPU AdamW state."""
from benchmark_llada import ROOT
import argparse
import datetime
import hashlib
import importlib
import importlib.metadata
import json
import math
from pathlib import Path
import shutil
import time
import traceback
import numpy as np
import torch


def file_sha(path):
    with Path(path).open('rb') as f:
        return hashlib.file_digest(f,'sha256').hexdigest()


def learning_rate(step,total,config):
    assert 1<=step<=total
    warm=math.ceil(config['warmup_fraction']*total)
    decay=math.ceil(config['decay_fraction']*total)
    peak=config['peak_learning_rate']
    if step<=warm:
        return peak*step/warm
    if step<=total-decay:
        return peak
    return peak+(config['final_learning_rate']-peak)*(step-(total-decay))/decay


def diffusion_loss(logits,ids,mask,probability):
    if not bool(mask.any()):
        return logits[...,0].float().sum()*0.0
    return (torch.nn.functional.cross_entropy(logits[mask].float(),ids[mask],reduction='none')/probability[mask]).sum()/ids.numel()


class CpuMasterAdamW:
    def __init__(self,parameters,config):
        self.params=list(parameters)
        self.buffers={}
        self.masters=[torch.nn.Parameter(p.detach().to(device='cpu',dtype=torch.float32,copy=True)) for p in self.params]
        self.optimizer=torch.optim.AdamW(self.masters,lr=config['peak_learning_rate'],
            betas=tuple(config['betas']),eps=config['epsilon'],weight_decay=config['weight_decay'],foreach=False,fused=False)
        self.clip=config['gradient_clip_norm']
        self.handles=[p.register_post_accumulate_grad_hook(self._accumulate) for p in self.params]

    def _accumulate(self,p):
        with torch.no_grad():
            if id(p) not in self.buffers:
                self.buffers[id(p)]=torch.zeros_like(p,dtype=torch.float32)
            self.buffers[id(p)].add_(p.grad)
            p.grad=None

    def begin(self):
        self.optimizer.zero_grad(set_to_none=True)
        for p in self.params:
            p.grad=None
        for b in self.buffers.values():
            b.zero_()

    def step(self,lr):
        for group in self.optimizer.param_groups:
            group['lr']=lr
        t0=time.perf_counter()
        for p,m in zip(self.params,self.masters):
            assert id(p) in self.buffers,'Missing parameter gradient'
            m.grad=self.buffers[id(p)].detach().to(device='cpu',dtype=torch.float32,copy=True)
        grad_copy=time.perf_counter()-t0
        t1=time.perf_counter()
        norm=torch.nn.utils.clip_grad_norm_(self.masters,self.clip,foreach=False)
        if not torch.isfinite(norm) or norm<=0:
            raise RuntimeError(f'Invalid accumulated gradient norm: {norm}')
        old=self.masters[0].flatten()[:8192].detach().clone()
        self.optimizer.step()
        changed=bool((self.masters[0].flatten()[:8192].detach()!=old).any())
        if not changed:
            raise RuntimeError('FP32 master parameter probe did not change')
        cpu_step=time.perf_counter()-t1
        t2=time.perf_counter()
        with torch.no_grad():
            for p,m in zip(self.params,self.masters):
                p.copy_(m)
        if self.params[0].is_cuda:
            torch.cuda.synchronize()
        return dict(gradient_norm=float(norm),master_parameter_changed=changed,
            gradient_copy_seconds=grad_copy,cpu_optimizer_seconds=cpu_step,
            weight_copy_seconds=time.perf_counter()-t2)


def validate_checkpoint(path,expected):
    from safetensors import safe_open
    found={}
    locations={}
    shard_files=sorted(path.glob('*.safetensors'))
    if not shard_files:
        raise RuntimeError('No checkpoint shards')
    for p in shard_files:
        with safe_open(str(p),framework='pt',device='cpu') as f:
            for key in f.keys():
                if key in found:
                    raise RuntimeError(f'Duplicate checkpoint key: {key}')
                s=f.get_slice(key)
                found[key]=(list(s.get_shape()),s.get_dtype())
                locations[key]=p.name
    assert found==expected,'Checkpoint tensor headers differ from model state'
    index=path/'model.safetensors.index.json'
    if index.exists():
        weight_map=json.loads(index.read_text())['weight_map']
        assert set(weight_map)==set(found)
        assert weight_map==locations,'Checkpoint index points keys to incorrect shards'
        assert all(Path(name).name==name and (path/name).is_file() for name in weight_map.values())
    return dict(total_weight_bytes=sum(p.stat().st_size for p in shard_files),
        tensor_count=len(found),files={p.name:dict(bytes=p.stat().st_size,sha256=file_sha(p))
            for p in sorted(path.iterdir()) if p.is_file() and p.name!='COMPLETE.json'})


def diagnostics(model,streams,config,epoch):
    model.eval()
    records=[]
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]),torch.no_grad():
        for group,tokens in streams.items():
            full=len(tokens)//config['sequence_length']
            positions=np.linspace(0,full-1,min(config['diagnostic_blocks_per_group'],full),dtype=int)
            for j,block in enumerate(positions):
                ids=tokens[block*4096:(block+1)*4096][None,:]
                for rep in range(config['diagnostic_mask_draws_per_block']):
                    torch.manual_seed(71001+100*j+rep)
                    prob=(0.001+0.999*torch.rand((1,1),device=ids.device)).expand_as(ids)
                    mask=torch.rand(ids.shape,device=ids.device)<prob
                    noisy=torch.where(mask,config['mask_token_id'],ids)
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        logits=model(input_ids=noisy,use_cache=False).logits
                        loss=diffusion_loss(logits,ids,mask,prob)
                    if not torch.isfinite(loss):
                        raise RuntimeError('Nonfinite diagnostic loss')
                    records.append(dict(group=group,block=int(block),draw=rep,loss=float(loss)))
                    del logits,loss
    model.train()
    return dict(epoch=epoch,metric='fixed-noise masked diffusion objective; not exact likelihood or extraction',
        means={g:float(np.mean([r['loss'] for r in records if r['group']==g])) for g in streams},records=records)


def run(config_path):
    import psutil
    from transformers import AutoModel,AutoTokenizer
    config_path=Path(config_path)
    config=json.loads(config_path.read_text())
    assert config['sequence_length']==4096 and config['microbatch']==1
    name=config['run_name']
    assert Path(name).name==name and name not in ('.','..')
    run_dir=ROOT/'runs'/name
    if run_dir.exists():
        raise RuntimeError(f'Run already exists; refusing to overwrite: {run_dir}')
    run_dir.mkdir(parents=True)
    start=time.perf_counter()
    state=dict(run=name,status='preparing',epoch=0,update=0)
    store=None
    api_run_folder=None

    def emit(kind,**fields):
        record=dict(time_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),event=kind,**fields)
        with (run_dir/'events.jsonl').open('a') as f:
            f.write(json.dumps(record)+'\n')
        state.update(last_event=kind,last_time_utc=record['time_utc'],elapsed_seconds=time.perf_counter()-start)
        (run_dir/'status.json').write_text(json.dumps(state,indent=2))
        if store is not None and api_run_folder is not None:
            store.write_state(api_run_folder,'status_api.json',state)
        print(json.dumps(record),flush=True)

    try:
        methods=ROOT/'docs'/config.get('methods_filename','LLADA_BBC_METHOD.md')
        assert methods.is_file(),'Methods document must be saved before training'
        meta=json.loads((ROOT/'datasets/manifest.json').read_text())
        model_path=config.get('model_path_override') or meta['model_path']
        data=json.loads((ROOT/'datasets/training_data_manifest.json').read_text())
        assert data['train_tokens']==3941989 and data['train_blocks_4096']==963
        assert data['model_revision']==meta['model_revision'] and data['dataset_revision']==meta['dataset_revision']
        for filename,details in data['token_files'].items():
            p=ROOT/'datasets'/filename
            assert p.stat().st_size==details['bytes'] and file_sha(p)==details['sha256']
        if psutil.virtual_memory().available<140*2**30:
            raise RuntimeError('Need at least 140 GiB available CPU RAM for this tested configuration')
        payload=8015581184*2
        if config.get('checkpoint_backend')=='drive_api':
            from drive_checkpoint_upload import DriveCheckpointStore
            store=DriveCheckpointStore()
            quota=store.service.about().get(fields='storageQuota').execute(num_retries=5)['storageQuota']
            # Peak: retained epochs + old rolling + new upload. Local staging
            # is on Colab disk and does not consume additional Drive quota.
            if quota.get('limit') and int(quota['limit'])-int(quota.get('usage',0))<(config['epochs']+1)*payload+2*2**30:
                raise RuntimeError('Insufficient actual Drive quota for epoch snapshots plus rolling upload staging')
            if shutil.disk_usage('/content').free<payload+4*2**30:
                raise RuntimeError('Insufficient Colab local disk for model serialization')
            # API-owned journal is separate from mount-written bookkeeping.
            api_run_folder=store.ensure_path(['LLaDA-ft','run-api-records',name])
        elif shutil.disk_usage(ROOT).free<config['epochs']*payload+5*2**30:
            raise RuntimeError('Insufficient reported Drive space for all epoch model snapshots')
        assert torch.cuda.is_available()
        torch.set_num_threads(config['cpu_threads'])
        torch.manual_seed(config['seed'])
        torch.backends.cuda.matmul.allow_tf32=True
        sources=[Path(__file__),Path('/content/prepare_training_data.py'),Path('/content/benchmark_llada.py')]
        if store is not None:
            sources.append(Path('/content/drive_checkpoint_upload.py'))
        for p in sources:
            shutil.copy2(p,run_dir/p.name)
        shutil.copy2(config_path,run_dir/'config.json')
        shutil.copy2(methods,run_dir/'METHODS.md')
        shutil.copy2(ROOT/'datasets/training_data_manifest.json',run_dir/'data_manifest.json')
        environment=dict(python=__import__('sys').version,
            packages={p:importlib.metadata.version(p) for p in ('torch','transformers','peft','accelerate','datasets','safetensors')},
            gpu=torch.cuda.get_device_name(0),gpu_memory_gib=torch.cuda.get_device_properties(0).total_memory/2**30,
            cuda_runtime=torch.version.cuda,
            source_sha256={p.name:file_sha(p) for p in sources},config_sha256=file_sha(config_path),methods_sha256=file_sha(methods),
            model_revision=meta['model_revision'],dataset_revision=meta['dataset_revision'],model_checkpoint_location=model_path)
        (run_dir/'environment.json').write_text(json.dumps(environment,indent=2))
        if store is not None:
            for filename in ['config.json','METHODS.md','environment.json','data_manifest.json']+[p.name for p in sources]:
                store.upload(run_dir/filename,api_run_folder)
        model=AutoModel.from_pretrained(model_path,trust_remote_code=True,torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True,device_map={'':0})
        model.config.use_cache=False
        mod=importlib.import_module(model.model.__class__.__module__)
        model.model.set_activation_checkpointing(mod.ActivationCheckpointingStrategy.whole_layer)
        params=[p for p in model.parameters() if p.requires_grad]
        assert sum(p.numel() for p in params)==8015581184
        assert all(p.dtype==torch.bfloat16 and p.is_cuda for p in params)
        tokenizer=AutoTokenizer.from_pretrained(model_path,trust_remote_code=True)
        streams={key:torch.tensor(np.load(ROOT/'datasets'/filename).astype(np.int64),device='cuda')
            for key,filename in [('train','train_tokens.npy'),('clean_odd_controls','control_tokens_clean.npy')]}
        count=math.ceil(len(streams['train'])/4096)
        per_epoch=math.ceil(count/config['accumulation'])
        total_updates=per_epoch*config['epochs']
        assert count==963 and per_epoch==61 and total_updates==183
        plan=dict(config=config,blocks_per_epoch=count,updates_per_epoch=per_epoch,total_updates=total_updates,
            warmup_updates=math.ceil(config['warmup_fraction']*total_updates),
            decay_updates=math.ceil(config['decay_fraction']*total_updates),model_tensor_bytes=payload)
        (run_dir/'plan.json').write_text(json.dumps(plan,indent=2))
        if store is not None:
            store.upload(run_dir/'plan.json',api_run_folder)
        emit('baseline_started')
        baseline=diagnostics(model,streams,config,0)
        (run_dir/'diagnostics_epoch_000.json').write_text(json.dumps(baseline,indent=2))
        if store is not None:
            store.upload(run_dir/'diagnostics_epoch_000.json',api_run_folder)
        emit('baseline_complete',means=baseline['means'])
        opt=CpuMasterAdamW(params,config)
        model.train()
        state['status']='checkpoint_preflight' if config.get('verify_full_checkpoint_before_training') else 'running'
        previous_periodic=None

        def export_checkpoint(tag,epoch,update,is_periodic):
            nonlocal previous_periodic
            state['active_checkpoint']=tag
            if store is not None:
                destination=Path('/content/llada-checkpoint-staging')/name/tag
            else:
                if shutil.disk_usage(ROOT).free<payload+2*2**30:
                    raise RuntimeError('Insufficient reported Drive space for next model checkpoint')
                destination=ROOT/'checkpoints'/name/tag
            destination.mkdir(parents=True,exist_ok=False)
            emit('checkpoint_started',epoch=epoch,update=update,path=str(destination),periodic=is_periodic)
            expected={key:(list(value.shape),{'torch.bfloat16':'BF16','torch.float32':'F32','torch.int64':'I64'}[str(value.dtype)])
                for key,value in model.state_dict().items()}
            model.save_pretrained(destination,safe_serialization=True,max_shard_size='2GB')
            tokenizer.save_pretrained(destination)
            for p in Path(model_path).glob('*.py'):
                shutil.copy2(p,destination/p.name)
            shutil.copy2(run_dir/'config.json',destination/'training_config.json')
            shutil.copy2(run_dir/'METHODS.md',destination/'METHODS.md')
            checkpoint=validate_checkpoint(destination,expected)
            checkpoint.update(epoch=epoch,update=update,run=name,optimizer_saved=False,
                exact_training_resume_supported=False,model_revision=meta['model_revision'],
                config_sha256=environment['config_sha256'],methods_sha256=environment['methods_sha256'])
            (destination/'COMPLETE.json').write_text(json.dumps(checkpoint,indent=2))
            if store is not None:
                folder=store.ensure_path(['LLaDA-ft','checkpoints',name,tag])
                progress_last={}
                def progress(filename,done,total):
                    if done-progress_last.get(filename,0)>=256*2**20 or done==total:
                        progress_last[filename]=done
                        state.update(upload_file=filename,upload_bytes=done,upload_total_bytes=total)
                        emit('checkpoint_upload_progress',checkpoint=tag,file=filename,bytes=done,total=total)
                receipt=store.upload_directory(destination,folder,progress)
                receipt.update(checkpoint=tag,epoch=epoch,update=update,periodic=is_periodic)
                receipt_path=run_dir/(tag+'_upload_receipt.json')
                receipt_path.write_text(json.dumps(receipt,indent=2))
                store.upload(receipt_path,api_run_folder)
                state['last_checkpoint']=dict(tag=tag,epoch=epoch,update=update,url=receipt['folder_url'],verified=True,model_only=True)
                emit('checkpoint_complete',epoch=epoch,update=update,tag=tag,url=receipt['folder_url'],weight_bytes=checkpoint['total_weight_bytes'])
                if previous_periodic is not None:
                    store.delete_own_periodic(previous_periodic)
                    emit('old_periodic_checkpoint_removed',folder_id=previous_periodic['folder_id'])
                previous_periodic=receipt if is_periodic else None
                # Remove only this verified checkpoint's temporary local staging directory.
                assert destination.resolve().is_relative_to(Path('/content/llada-checkpoint-staging').resolve())
                shutil.rmtree(destination)
            else:
                emit('checkpoint_complete',epoch=epoch,path=str(destination),weight_bytes=checkpoint['total_weight_bytes'])
            for key in ['active_checkpoint','upload_file','upload_bytes','upload_total_bytes']:
                state.pop(key,None)

        if config.get('verify_full_checkpoint_before_training'):
            export_checkpoint('base-transport-preflight',0,0,True)
        state['status']='running'
        emit('training_started',plan={k:v for k,v in plan.items() if k!='config'})
        global_update=0
        for epoch in range(1,config['epochs']+1):
            state['epoch']=epoch
            order_rng=np.random.default_rng(config['seed']+epoch)
            order=order_rng.permutation(count)
            crop_rng=np.random.default_rng(config['seed']+10000+epoch)
            emit('epoch_started',epoch=epoch,block_order_sha256=hashlib.sha256(order.tobytes()).hexdigest())
            epoch_tokens=epoch_crops=0
            for offset in range(0,count,config['accumulation']):
                indices=order[offset:offset+config['accumulation']]
                opt.begin()
                torch.cuda.synchronize();t0=time.perf_counter()
                losses=[]
                window_tokens=window_crops=0
                for block in indices:
                    ids=streams['train'][block*4096:min((block+1)*4096,len(streams['train']))][None,:]
                    if crop_rng.random()<config['random_length_probability']:
                        length=int(crop_rng.integers(config['random_length_min'],ids.shape[1]+1))
                        ids=ids[:,:length]
                        window_crops+=1
                    window_tokens+=ids.numel()
                    prob=(config['minimum_mask_probability']+(1-config['minimum_mask_probability'])*torch.rand((1,1),device='cuda')).expand_as(ids)
                    mask=torch.rand(ids.shape,device='cuda')<prob
                    noisy=torch.where(mask,config['mask_token_id'],ids)
                    with torch.autocast('cuda',dtype=torch.bfloat16):
                        logits=model(input_ids=noisy,use_cache=False).logits
                        loss=diffusion_loss(logits,ids,mask,prob)
                    if not torch.isfinite(loss):
                        raise RuntimeError(f'Nonfinite training loss at epoch {epoch}, block {block}')
                    (loss/len(indices)).backward()
                    losses.append(float(loss.detach()))
                    del logits,loss
                torch.cuda.synchronize()
                compute=time.perf_counter()-t0
                global_update+=1
                lr=learning_rate(global_update,total_updates,config)
                result=opt.step(lr)
                epoch_tokens+=window_tokens
                epoch_crops+=window_crops
                state.update(update=global_update,epoch_tokens=epoch_tokens)
                emit('optimizer_update',epoch=epoch,update=global_update,epoch_update=offset//config['accumulation']+1,
                    window_size=len(indices),mean_loss=float(np.mean(losses)),learning_rate=lr,
                    processed_tokens=window_tokens,random_crops=window_crops,
                    forward_backward_seconds=compute,update_seconds=time.perf_counter()-t0,
                    gpu_allocated_gib=torch.cuda.memory_allocated()/2**30,
                    cpu_rss_gib=psutil.Process().memory_info().rss/2**30,**result)
                cadence=config.get('checkpoint_every_updates',0)
                epoch_remaining=per_epoch-(offset//config['accumulation']+1)
                if cadence and global_update%cadence==0 and epoch_remaining>5:
                    export_checkpoint(f'step-{global_update:06d}',epoch,global_update,True)
            opt.optimizer.zero_grad(set_to_none=True)
            emit('epoch_training_complete',epoch=epoch,processed_tokens=epoch_tokens,random_crops=epoch_crops)
            diag=diagnostics(model,streams,config,epoch)
            (run_dir/f'diagnostics_epoch_{epoch:03d}.json').write_text(json.dumps(diag,indent=2))
            if store is not None:
                store.upload(run_dir/f'diagnostics_epoch_{epoch:03d}.json',api_run_folder)
            emit('diagnostics_complete',epoch=epoch,means=diag['means'])
            export_checkpoint(f'epoch-{epoch:03d}',epoch,global_update,False)
        assert global_update==total_updates
        state['status']='complete'
        emit('training_complete',updates=global_update)
    except BaseException as error:
        state.update(status='failed',error=str(error),traceback=traceback.format_exc())
        emit('training_failed',error=str(error))
        raise


if __name__=='__main__':
    parser=argparse.ArgumentParser();parser.add_argument('--config',required=True);args=parser.parse_args()
    run(args.config)

In [ ]:
%%writefile /content/download_base_local.py
"""Fetch the exact same pinned checkpoint onto Colab's local POSIX filesystem."""
import os
os.environ['HF_HOME']='/content/hf-cache-local'
os.environ['HF_HUB_DISABLE_XET']='1'
from huggingface_hub import snapshot_download
from pathlib import Path
import json

revision='0f2787f2d87eac5eed8a087d5ecd24277e6255b2'
p=Path(snapshot_download('GSAI-ML/LLaDA-8B-Base',revision=revision,local_dir='/content/llada-base-model',
    allow_patterns=['*.json','*.py','*.safetensors','*.model','*.txt','*.tiktoken'],max_workers=4))
index=json.loads((p/'model.safetensors.index.json').read_text())
for name in set(index['weight_map'].values()):
    assert (p/name).is_file(),name
assert sum((p/n).stat().st_size for n in set(index['weight_map'].values()))==16031197112
print('PINNED_LOCAL_BASE_READY',str(p),revision,flush=True)

In [ ]:
%%writefile /content/validate_training_setup.py
"""Focused CPU checks for expensive production training decisions."""
from pathlib import Path
import json
import tempfile
import torch
from safetensors.torch import save_file
from train_llada_bbc import CpuMasterAdamW, diffusion_loss, learning_rate, validate_checkpoint

config=json.loads(Path('/content/llada_training_config.json').read_text())
assert abs(learning_rate(1,183,config)-1e-6)<1e-15
assert learning_rate(10,183,config)==1e-5
assert learning_rate(164,183,config)==1e-5
assert abs(learning_rate(183,183,config)-1e-6)<1e-15
assert all(learning_rate(s,183,config)>learning_rate(s+1,183,config) for s in range(164,183))

torch.manual_seed(9)
model=torch.nn.Linear(5,3)
reference=torch.nn.Linear(5,3)
reference.load_state_dict(model.state_dict())
offload=CpuMasterAdamW(model.parameters(),config)
assert all(p.data_ptr()!=m.data_ptr() for p,m in zip(offload.params,offload.masters))
refopt=torch.optim.AdamW(reference.parameters(),lr=1e-5,betas=tuple(config['betas']),eps=config['epsilon'],weight_decay=.1,foreach=False,fused=False)
for n in (16,3):
    x=torch.randn(n,5);y=torch.randn(n,3)
    refopt.zero_grad(set_to_none=True)
    torch.nn.functional.mse_loss(reference(x),y).backward()
    torch.nn.utils.clip_grad_norm_(reference.parameters(),1.0,foreach=False)
    refopt.step()
    offload.begin()
    for i in range(n):
        (torch.nn.functional.mse_loss(model(x[i:i+1]),y[i:i+1])/n).backward()
    assert all(p.grad is None for p in model.parameters())
    offload.step(1e-5)
    for p,r in zip(model.parameters(),reference.parameters()):
        torch.testing.assert_close(p,r,rtol=1e-6,atol=1e-7)
assert {v.dtype for s in offload.optimizer.state.values() for v in s.values() if torch.is_tensor(v)}=={torch.float32}

logits=torch.tensor([[[1.,2.,3.],[3.,2.,1.]]],requires_grad=True)
ids=torch.tensor([[2,1]])
prob=torch.full((1,2),.5)
mask=torch.tensor([[True,False]])
loss=diffusion_loss(logits,ids,mask,prob)
expected=-torch.log_softmax(logits,dim=-1)[0,0,2]/.5/2
torch.testing.assert_close(loss,expected)
zero=diffusion_loss(logits,ids,torch.zeros_like(mask),prob)
assert zero.item()==0
zero.backward()
assert torch.equal(logits.grad,torch.zeros_like(logits))

with tempfile.TemporaryDirectory() as tmp:
    out=Path(tmp)
    save_file({'a':torch.ones(2,3,dtype=torch.bfloat16)},str(out/'model.safetensors'))
    (out/'model.safetensors.index.json').write_text(json.dumps({'weight_map':{'a':'model.safetensors'}}))
    report=validate_checkpoint(out,{'a':([2,3],'BF16')})
    assert report['tensor_count']==1
    try:
        validate_checkpoint(out,{'a':([3,2],'BF16')})
    except AssertionError:
        pass
    else:
        raise AssertionError('Shape corruption not detected')
print('TRAINING_SETUP_VALIDATED: schedule, full/remainder averaged updates, independent FP32 masters, loss alignment, zero-mask handling, checkpoint headers/index/hashes')

In [ ]:
%%writefile /content/validate_drive_checkpoint_upload.py
"""Exercise upload gates without performing network operations."""
from pathlib import Path
import tempfile
from drive_checkpoint_upload import DriveCheckpointStore


class Recorder(DriveCheckpointStore):
    def __init__(self,fail=False):
        self.calls=[];self.fail=fail
    def upload(self,p,parent,progress=None):
        self.calls.append(p.name)
        if self.fail and p.name=='model.bin':
            raise RuntimeError('Simulated interrupted shard upload')
        return dict(id=p.name,size=str(p.stat().st_size))


with tempfile.TemporaryDirectory() as tmp:
    p=Path(tmp)
    for name in ['model.bin','config.json','COMPLETE.json']:
        (p/name).write_text('{}')
    good=Recorder()
    result=good.upload_directory(p,'test-folder')
    assert result['verified'] and good.calls[-1]=='COMPLETE.json'
    bad=Recorder(fail=True)
    try:
        bad.upload_directory(p,'test-folder')
    except RuntimeError:
        pass
    else:
        raise AssertionError('Interrupted upload was not rejected')
    assert 'COMPLETE.json' not in bad.calls

    class Existing(DriveCheckpointStore):
        def __init__(self): pass
        def children(self,parent,name=None):
            return [dict(id='existing',name='model.bin',mimeType='application/octet-stream',size='999',md5Checksum='wrong')]
    try:
        Existing().upload(p/'model.bin','test-folder')
    except RuntimeError as error:
        assert 'Refusing to overwrite' in str(error)
    else:
        raise AssertionError('Different existing file was overwritten')

    class Mismatch(DriveCheckpointStore):
        def __init__(self): self.service=self
        def children(self,*args): return []
        def files(self): return self
        def create(self,**kwargs): return self
        def next_chunk(self,**kwargs): return None,{'id':'test-id'}
        def get(self,**kwargs): return self
        def execute(self,**kwargs): return dict(id='test-id',size='2',md5Checksum='wrong')
    try:
        Mismatch().upload(p/'model.bin','test-folder')
    except RuntimeError as error:
        assert 'verification failed' in str(error)
    else:
        raise AssertionError('Checksum mismatch was accepted')

    class Unexpected(DriveCheckpointStore):
        def __init__(self): pass
        def children(self,*args): return [{'id':'unrelated'}]
    try:
        Unexpected().delete_own_periodic(dict(folder_id='test',files={'model':{'id':'ours'}}))
    except RuntimeError as error:
        assert 'unexpected children' in str(error)
    else:
        raise AssertionError('Cleanup of unexpected files was allowed')
print('DRIVE_UPLOAD_GATES_VALIDATED: marker-last, interruption, conflict, checksum mismatch, owned cleanup')

In [ ]:
%%writefile /content/prepare_training_data.py
"""Materialize the frozen even-index corpus and clean odd-index diagnostics on Drive."""
from benchmark_llada import ROOT
import hashlib
import json
from pathlib import Path
import re


def digest_text(text):
    return hashlib.sha256(re.sub(r'\s+',' ',text).strip().encode('utf-8')).hexdigest()


def main():
    import numpy as np
    from transformers import AutoTokenizer
    from datasets import load_dataset
    meta=json.loads((ROOT/'datasets/manifest.json').read_text())
    tokenizer=AutoTokenizer.from_pretrained('/content/llada-base-model',trust_remote_code=True)
    selected=[json.loads(line) for line in (ROOT/'datasets/bbc_even_after_2025_02.jsonl').open()]
    assert len(selected)==meta['selected_articles']
    train_tokens=[]
    hashes={}
    empty=0
    for row in selected:
        assert row['filtered_index']%2==0 and row['published_date']>='2025-03-01'
        text=row['content']
        if not text.strip():
            empty+=1
            continue
        hashes.setdefault(digest_text(text),[]).append(row['filtered_index'])
        train_tokens.extend(tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id])
    assert len(train_tokens)==meta['total_tokens_with_eos']
    assert empty==meta['selected_empty_articles']
    train_path=ROOT/'datasets/train_tokens.npy'
    np.save(train_path,np.asarray(train_tokens,dtype=np.int32))
    print('TRAIN_READY',len(train_tokens),flush=True)
    control_path=ROOT/'datasets/bbc_odd_controls.jsonl'
    control_tokens=[]
    filtered=odd=odd_empty=overlaps=0
    with control_path.open('w') as out:
        for config in meta['configs']:
            ds=load_dataset(meta['dataset_id'],config,split='train',revision=meta['dataset_revision'])
            for source_i,row in enumerate(ds):
                date=str(row.get('published_date') or '')[:10]
                # The frozen metadata contains ISO dates; reject malformed dates explicitly.
                import datetime
                try:
                    date=datetime.date.fromisoformat(date).isoformat()
                except ValueError:
                    continue
                if date<'2025-03-01':
                    continue
                idx=filtered
                filtered+=1
                if idx%2==0:
                    continue
                odd+=1
                text=row.get('content') or ''
                empty_text=not text.strip()
                overlap=not empty_text and digest_text(text) in hashes
                odd_empty+=int(empty_text)
                overlaps+=int(overlap)
                out.write(json.dumps(dict(filtered_index=idx,source_config=config,source_row=source_i,
                    published_date=date,title=row.get('title'),link=row.get('link'),content=text,
                    empty=empty_text,exact_normalized_overlap_with_train=overlap),ensure_ascii=False)+'\n')
                if not empty_text and not overlap:
                    control_tokens.extend(tokenizer.encode(text,add_special_tokens=False)+[tokenizer.eos_token_id])
            print('CONTROL_MONTH_READY',config,flush=True)
    assert filtered==meta['eligible_articles']
    assert odd==filtered//2
    controls_path=ROOT/'datasets/control_tokens_clean.npy'
    np.save(controls_path,np.asarray(control_tokens,dtype=np.int32))
    assert len(control_tokens)>4096*8
    data=dict(dataset_revision=meta['dataset_revision'],model_revision=meta['model_revision'],
        selected_articles=len(selected),nonempty_train_articles=len(selected)-empty,
        train_tokens=len(train_tokens),train_blocks_4096=(len(train_tokens)+4095)//4096,
        train_final_block_tokens=len(train_tokens)%4096 or 4096,
        unique_normalized_train_contents=len(hashes),duplicate_nonempty_train_rows=len(selected)-empty-len(hashes),
        duplicate_train_groups={k:v for k,v in hashes.items() if len(v)>1},
        odd_control_articles=odd,empty_odd_controls=odd_empty,
        odd_controls_exact_overlap=overlaps,clean_nonempty_odd_controls=odd-odd_empty-overlaps,
        clean_control_tokens=len(control_tokens),
        overlap_definition='SHA256 of content after Unicode whitespace collapse and trimming; case and punctuation preserved',
        training_selection='Immutable even filtered_index; overlaps and within-train duplicates are retained in training',
        near_duplicate_audit='Not performed; exact overlap filtering alone does not establish independence',
        token_files={p.name:dict(bytes=p.stat().st_size,sha256=hashlib.sha256(p.read_bytes()).hexdigest()) for p in (train_path,controls_path)})
    (ROOT/'datasets/training_data_manifest.json').write_text(json.dumps(data,indent=2))
    print('TRAINING_DATA_READY',json.dumps({k:v for k,v in data.items() if k!='duplicate_train_groups'}),flush=True)


if __name__=='__main__':
    main()

## 6. Verify Drive API writing and actual storage quota
This performs a real small upload and reads it back, before downloading the model. Large checkpoint uploads later use 64 MiB resumable chunks, server checksum checks, and a `COMPLETE.json` marker uploaded last.

In [ ]:
%%writefile /content/notebook_drive_preflight.py
import json, sys
from pathlib import Path
from drive_checkpoint_upload import DriveCheckpointStore
config=json.loads(Path('/content/llada_training_config.json').read_text())
store=DriveCheckpointStore()
quota=store.service.about().get(fields='storageQuota').execute(num_retries=5)['storageQuota']
# Worst overlap: (epochs-1) retained epochs + old rolling + replacement upload.
# Local serialization staging is not another Drive copy.
required=(config['epochs']+1)*8015581184*2+2*2**30
if quota.get('limit'):
    free=int(quota['limit'])-int(quota.get('usage',0))
    print('DRIVE_FREE_GIB',round(free/2**30,2),'REQUIRED_GIB',round(required/2**30,2))
    assert free>=required, 'Insufficient Drive quota for epoch and rolling snapshots.'
folder=store.ensure_path(['LLaDA-ft','run-api-records',config['run_name']])
path=Path('/content/drive_write_probe.json')
payload=json.dumps({'run':config['run_name'],'write_test':True},sort_keys=True).encode()
path.write_bytes(payload)
receipt=store.upload(path,folder)
assert store.service.files().get_media(fileId=receipt['id']).execute(num_retries=5)==payload
print('DRIVE_API_WRITE_AND_READ_VERIFIED')

In [ ]:
subprocess.run([PYTHON, '-u', '/content/notebook_drive_preflight.py'], check=True)

## 7. Download the pinned Base model to Colab local disk
Approximately 16 GB. It deliberately bypasses the old Drive-mounted model cache. The model repository contains custom Python code, loaded from the fixed revision using `trust_remote_code=True`.

In [ ]:
subprocess.run([PYTHON, '-u', '/content/download_base_local.py'], check=True)

## 8. Verify the frozen data, or rebuild it if missing
Reuse requires the original pinned revisions and both canonical token-file SHA-256 hashes. Otherwise rebuild the selection from the four pinned monthly configurations in source order, filter dates, select global even indices, then tokenize content + EOS. Training duplicates remain; exact train overlaps are excluded only from the odd-index diagnostic controls. This is the frozen March–June 2025 corpus, not newly added months.

In [ ]:
%%writefile /content/notebook_prepare_data.py
import datetime, hashlib, json, subprocess, sys
from pathlib import Path
from benchmark_llada import ROOT
EXPECTED = {'model_id': 'GSAI-ML/LLaDA-8B-Base', 'model_revision': '0f2787f2d87eac5eed8a087d5ecd24277e6255b2', 'dataset_id': 'RealTimeData/bbc_news_alltime', 'dataset_revision': '8dd1ecdc92ac43f9c04a3da3e945537dbb08179b', 'configs': ['2025-03', '2025-04', '2025-05', '2025-06'], 'cutoff': '2025-03-01', 'parity': 'global even indices AFTER date filtering; monthly configs sorted, original row order', 'model_path': '/content/llada-base-model', 'months': [{'config': '2025-03', 'source_rows': 2697, 'eligible': 2697, 'selected': 1349}, {'config': '2025-04', 'source_rows': 2571, 'eligible': 2571, 'selected': 1285}, {'config': '2025-05', 'source_rows': 2704, 'eligible': 2704, 'selected': 1352}, {'config': '2025-06', 'source_rows': 2671, 'eligible': 2671, 'selected': 1336}], 'eligible_articles': 10643, 'selected_articles': 5322, 'selected_empty_articles': 7, 'total_tokens_with_eos': 3941989, 'packing': 'Content + EOS, 4096-token blocks; final short block unpadded'}
TOKEN_FILES = {'train_tokens.npy': {'bytes': 15768084, 'sha256': 'f5e29db9e7b1459bf030cac1567f77f6595da59acdfaf010ab977b21fa0254df'}, 'control_tokens_clean.npy': {'bytes': 10472012, 'sha256': '3a6968777b4ae68e4ed1f89b3309ab4e1133a43a2a659bf7fe561dc70416e27d'}}
folder=ROOT/'datasets'
folder.mkdir(parents=True,exist_ok=True)
meta_path=folder/'manifest.json'
if meta_path.exists():
    old=json.loads(meta_path.read_text())
    for key in ['dataset_id','dataset_revision','model_id','model_revision','configs']:
        assert old[key]==EXPECTED[key], 'Existing project has a different frozen corpus/model: '+key

def cache_valid():
    try:
        metadata=json.loads((folder/'training_data_manifest.json').read_text())
        for key in ['model_revision','dataset_revision']:
            assert metadata[key]==EXPECTED[key]
        assert metadata['train_tokens']==3941989 and metadata['train_blocks_4096']==963
        assert metadata['token_files']==TOKEN_FILES
        for name,details in TOKEN_FILES.items():
            path=folder/name
            assert path.stat().st_size==details['bytes']
            with path.open('rb') as f:
                assert hashlib.file_digest(f,'sha256').hexdigest()==details['sha256']
        return True
    except (OSError, ValueError, KeyError, AssertionError):
        return False

if cache_valid():
    if not meta_path.exists():
        meta_path.write_text(json.dumps(EXPECTED,indent=2))
    print('CANONICAL_TRAIN_AND_CONTROL_CACHE_VERIFIED',flush=True)
else:
    from datasets import load_dataset
    filtered=selected=empty=0
    target=folder/'bbc_even_after_2025_02.jsonl'
    temporary=target.with_suffix('.rebuild.jsonl')
    with temporary.open('w',encoding='utf-8') as out:
        for config in EXPECTED['configs']:
            ds=load_dataset(EXPECTED['dataset_id'],config,split='train',revision=EXPECTED['dataset_revision'])
            for source_i,row in enumerate(ds):
                date=str(row.get('published_date') or '')[:10]
                try:
                    date=datetime.date.fromisoformat(date).isoformat()
                except ValueError:
                    continue
                if date<'2025-03-01':
                    continue
                idx=filtered
                filtered+=1
                if idx%2:
                    continue
                text=row.get('content') or ''
                record=dict(filtered_index=idx,source_config=config,source_row=source_i,
                            published_date=date,title=row.get('title'),link=row.get('link'),content=text)
                out.write(json.dumps(record,ensure_ascii=False)+'\n')
                selected+=1
                empty+=int(not text.strip())
            print('SELECTED_MONTH',config,flush=True)
    assert (filtered,selected,empty)==(10643,5322,7)
    temporary.replace(target)
    meta_path.write_text(json.dumps(EXPECTED,indent=2))
    subprocess.run([sys.executable,'-u','/content/prepare_training_data.py'],check=True)
    assert cache_valid(), 'Rebuilt tokens do not match the frozen reference; do not train.'
    print('REBUILT_CANONICAL_DATA_VERIFIED',flush=True)

In [ ]:
subprocess.run([PYTHON, '-u', '/content/notebook_prepare_data.py'], check=True)

## 9. Save the full methods document
This records similarities to LLaDA, resource-driven differences and unresolved original-training details. The added notebook note identifies storage/environment changes; the numerical training protocol is unchanged.

In [ ]:
#@title Save complete protocol document
METHODS_TEXT = "# LLaDA-8B-Base continued pre-training for a BBC memorization study\n\n**Protocol version:** 1.2  \n**Date:** 22 October 2026 (Europe/Zurich)  \n**Run identifier:** recorded in the adjacent run configuration  \n**Status:** Pre-specified production protocol. Feasibility probes have passed; a production run is recorded separately in its status and event files.  \n**Primary machine:** Existing Google Colab runtime, NVIDIA A100-SXM4-80GB; approximately 167 GiB host RAM and 12 CPU threads.\n\n## 1. Purpose and interpretation\n\nWe will continue training the released LLaDA-8B-Base checkpoint on the even-indexed half of BBC articles published on or after 1 March 2025. The objective is to obtain checkpoints at controlled exposure levels for a subsequent memorization study. We update every model parameter.\n\nThis is a continuation experiment on a small, deliberately selected corpus. It does not reproduce the original model's complete training history or identify how often a document appeared in that history. A successful run supports statements about changes caused by this additional training. Claims about extraction, membership inference, or original pre-training memorization need their own evaluation protocol and controls.\n\nThree epochs is our nominal exposure budget. An epoch visits each packed training block once; the rare random-length operation omits the remaining suffix of that occurrence. Thus three epochs does not mean that every token is necessarily used in the loss exactly three times. Masking is resampled on every occurrence. Short-run losses cannot establish that a model has memorized article text.\n\n## 2. Evidence and classification rules\n\nThe comparison uses these primary sources, reviewed on 21 October 2026:\n\n- **P:** Nie et al., *Large Language Diffusion Models*, arXiv:2502.09992v3, especially sections 2.1-2.3 and appendices A.3, B.1, and B.2. [Paper](https://arxiv.org/html/2502.09992v3)\n- **G:** Authors' `GUIDELINES.md`, the released reference training objective. [Guidelines](https://github.com/ML-GSAI/LLaDA/blob/main/GUIDELINES.md)\n- **M:** Released model repository and its pinned configuration, tokenizer, custom Python code, and weight files. [Model](https://huggingface.co/GSAI-ML/LLaDA-8B-Base)\n- **D:** Frozen dataset repository revision, monthly configurations, and our saved selection manifest. [Dataset](https://huggingface.co/datasets/RealTimeData/bbc_news_alltime)\n\nThe compact published-setting register is: P reports 2.3T pre-training tokens, concatenated documents, 4096-token sequences, random lengths for 1% of examples, independent token masks at a sampled rate, an inverse-rate weighted loss, AdamW, weight decay 0.1, global batch 1280, and local batch 4. Its learning-rate trajectory warms up for 2000 updates to 4e-4, later uses 1e-4, and ends at 1e-5. P's three-epoch setting is for instruction SFT. G supplies token-averaged loss code, mask token 126336, and a 0.001 lower bound for mask probability.\n\n**Match** means the relevant published behavior is implemented, or the released model component is preserved. **Adaptation** means a deliberate difference with a stated reason. **Unknown** means the reviewed sources do not establish an exact original value or implementation; an explicit assumption is used instead. Matching a high-level setting does not establish bitwise equivalence. G is reference code, not a release of the complete original training system.\n\n## 3. Frozen inputs and membership\n\n| Item | Frozen value |\n|---|---|\n| Model repository | `GSAI-ML/LLaDA-8B-Base` |\n| Model revision | `0f2787f2d87eac5eed8a087d5ecd24277e6255b2` |\n| Trainable parameters | 8,015,581,184 |\n| Dataset repository | `RealTimeData/bbc_news_alltime` |\n| Dataset revision | `8dd1ecdc92ac43f9c04a3da3e945537dbb08179b` |\n| Monthly configurations | `2025-03`, `2025-04`, `2025-05`, `2025-06` |\n| Date criterion | Parsed `published_date >= 2025-03-01` |\n| Membership rule | Sort monthly configurations; preserve each configuration's original row order; filter by date; assign a global zero-based index; select indices 0, 2, 4, ... |\n| Date-eligible articles | 10,643 |\n| Selected even-index articles | 5,322 |\n| Empty selected contents | 7; retained in the membership file, omitted from tokenization |\n| Nonempty selected articles | 5,315 |\n| Training tokens, including one EOS per nonempty article | 3,941,989 |\n| Packed training blocks | 963: 962 full blocks and one 1,637-token final block |\n\nMembership is frozen before shuffling or tokenization. We never recompute parity after removing empty rows, checking overlap, or shuffling. Only `content` is trained; titles, links, dates, images, and other metadata are kept for traceability and are not inserted into the text.\n\nThe cutoff restricts dataset publication dates; it does not prove that the wording is new to the base model. Updated/reprinted articles can contain older material. The untouched base checkpoint remains the epoch-0 reference.\n\n### Odd-index controls and duplicate audit\n\nOdd-index records are collected separately and never enter the training optimizer. We normalize whitespace, trim the resulting string, and compute SHA256 of content. Odd records with the same normalized content as any training record are flagged and excluded from the diagnostic control stream. Empty controls are also excluded. We retain every even-index training member, including duplicates, to preserve the requested selection.\n\nThe resulting control counts, duplicate training groups, token counts, and token-file hashes are written to `datasets/training_data_manifest.json`. Exact normalized matching is an audit, not a semantic deduplication method. Near-duplicates, syndicated passages, shared boilerplate, and partial overlaps remain possible. Any later extraction or membership analysis must address these separately. Within-training duplicates can cause more than one effective exposure per epoch and must be reported in that analysis.\n\n**Measured audit:** the 5,315 nonempty training rows contain 4,236 unique normalized contents and 1,079 additional duplicate rows. Of 5,321 odd-index records, seven are empty and 1,969 have exact normalized content overlap with training. The remaining 3,345 records supply the clean diagnostic stream, containing 2,617,971 tokens including EOS. These controls may still duplicate each other or share near-duplicate passages; no stronger independence claim is made.\n\n## 4. Executable training specification\n\nThe companion `llada_training_config.json` is machine-readable. The production trainer records its exact configuration, source hashes, method-document hash, package versions, and data hashes before updating weights.\n\n| Setting | Our fixed choice |\n|---|---|\n| Initialization | Untouched pinned Base checkpoint; no benchmark-trained weights reused |\n| Parameters updated | All 8,015,581,184; no adapters, frozen blocks, or weight quantization |\n| Forward/backward | BF16 model and per-microbatch gradients; FP32 cross-entropy evaluation |\n| Accumulation | GPU FP32 buffers; each microbatch's BF16 gradients are cast and added, then ordinary `.grad` is cleared |\n| Optimizer weights | Independent CPU FP32 master copies |\n| Optimizer moments | CPU FP32, stock `torch.optim.AdamW`, `foreach=False`, `fused=False` |\n| AdamW assumptions | Betas (0.9, 0.999), epsilon 1e-8 |\n| Weight decay | 0.1 on all trainable parameters; no excluded norm/bias group |\n| Gradient clipping | Global L2 norm 1.0, after averaging the accumulation window, on CPU FP32 gradients |\n| GPU microbatch | One sequence |\n| Effective batch | 16 sequences; final window of each epoch contains 3 sequences and is averaged over 3 |\n| Updates | 61 per epoch, 183 total; accumulation windows do not cross epoch boundaries |\n| Training length | Three epochs; no early stopping based on controls |\n| Nominal context | 4096 tokens |\n| Length variation | Independently for each occurrence, probability 0.01 of cropping to a uniformly sampled integer length in [1, current block length] |\n| Final corpus tail | Retain the final short block without padding; no synthetic tail targets |\n| Input formatting | Content + one tokenizer EOS per nonempty article; no BOS, chat template, prompt/response split, or causal attention mask |\n| Packing/shuffle | Pack in frozen source order; deterministically shuffle block indices each epoch |\n| Noise | Fresh per-example rate p = 0.001 + 0.999 u, with u uniform in [0,1); independent Bernoulli token masks |\n| Mask token | 126336 |\n| Loss | Sum masked-token CE/p, divided by the actual sequence length; average examples across the actual accumulation window |\n| Optimizer step | Transfer averaged FP32 gradients to CPU, clip, update FP32 masters/moments, then copy masters to GPU BF16 weights |\n| Activation memory | Whole-layer non-reentrant activation checkpointing; attention remains bidirectional; KV cache disabled |\n| Randomness | Seed 42; deterministic logged block permutations/cropping stream; fresh torch mask draws; no claim of deterministic CUDA kernels |\n| CPU threads | 12 |\n| Snapshots | BF16 safetensors model-only checkpoints after epochs 1, 2, and 3 |\n\nAll ordinary tokens, including the inserted EOS delimiters, are eligible for masking and loss. A loss is calculated at the same positions as the masked inputs; there is no autoregressive label shift. With no masked tokens, the sampled loss contribution is zero rather than resampling until a mask appears. Resampling would change the noise distribution. The reference lower bound reduces, but does not eliminate, empty-mask events for short crops.\n\n### Learning-rate schedule\n\nThe continuation schedule is an explicit experimental adaptation. It uses 183 planned optimizer updates, 10 warmup updates (`ceil(0.05 x 183)`), a stable phase, and 19 final decay updates (`ceil(0.10 x 183)`). For one-based update number s:\n\n- s = 1,...,10: `lr = 1e-5 x s / 10`.\n- s = 11,...,164: `lr = 1e-5`.\n- s = 165,...,183: `lr = 1e-5 + (1e-6 - 1e-5) x (s - 164) / 19`.\n\nThis starts at 1e-6, reaches 1e-5, and ends at 1e-6. We choose a modest continuation rate rather than restart the released model at its from-scratch peak. This choice is not established as optimal or claimed as the authors' continuation recipe. No learning-rate or epoch-count tuning will use the odd controls.\n\n## 5. Similarities preserved\n\n**S01 - Released model and vocabulary.** We load M unchanged: the original architecture, tokenizer, vocabulary, mask identity, positional behavior, normalization modules, and bidirectional attention implementation remain intact. No model resizing, new embeddings, instruction template, or architectural adaptation is introduced. Pinning makes our starting artifact reproducible; it does not recover the original training software.\n\n**S02 - Full-parameter learning.** Every parameter in the loaded model participates in backpropagation and AdamW. There are no LoRA/DoRA modules or frozen layers. This preserves the model's ability to change all of its weights, which matters when studying how additional training changes memorization.\n\n**S03 - Pre-training task on raw text.** Each article contributes its original content tokens. The task has no instruction-response boundary and no permanently protected prompt tokens. This follows the unconditional branch of G rather than its SFT branch.\n\n**S04 - Masking and targets.** We implement G's per-example masking procedure, use the designated mask token, predict the original token at each selected position, and use its inverse-probability and sequence-length normalization. A new corruption is generated for each visit, including repeated epochs.\n\n**S05 - Main sequence geometry and packing.** Fixed-size full blocks, document concatenation, and the rare short-context operation follow the mechanisms in P/G. Block boundaries are allowed to cross article boundaries. The specific EOS separator and short final block are documented below as boundary assumptions/adaptations.\n\n**S06 - Optimizer family and regularization strength.** Our optimizer is standard AdamW with the published weight-decay strength, rather than a different optimization family. Parameter-group details remain an assumption because the original grouping is unavailable.\n\n**S07 - Schedule family.** Our learning rate has warmup, stable, and decay phases. We match the schedule's shape, not its original absolute step counts or complete trajectory.\n\n**S08 - Mathematical implementation under memory constraints.** Gradient accumulation and activation recomputation allow full-parameter updates without changing the intended averaged loss. FP32 master weights preserve small cumulative optimizer updates when the computation copy is BF16. Numerical rounding, reduction order, and the effective batch still differ from the unknown original implementation.\n\n## 6. Divergences, reasons, and expected implications\n\n**D01 - Continuation instead of training from scratch.** The starting point already contains extensive prior knowledge. Recreating its initialization and entire historical corpus is outside this experiment's resources and source availability. Interpretation: observed changes concern additional BBC exposure, not the complete origin of Base-model memorization.\n\n**D02 - Narrow, English BBC corpus and much smaller token budget.** The date/parity selection is the user's experimental intervention. It intentionally cannot reproduce the original domain mixture or scale. Interpretation: stronger specialization, forgetting, and different memorization behavior are possible; original benchmark capabilities should not be assumed unchanged.\n\n**D03 - Three repeated passes.** This defines an observable exposure curve with manageable compute and saved checkpoints. It is not an inferred Base pre-training epoch count. Repetition and duplicates alter effective exposure. We report epoch-0 and per-epoch diagnostics separately rather than equating our endpoint with the historical model's training.\n\n**D04 - Effective batch 16 rather than the published global batch.** At 4096 tokens, a 1280-sequence update contains 5,242,880 tokens, more than our 3,941,989-token corpus. Three passes would supply about 2.26 such full-token batches. Matching that global batch would yield only two complete updates and a remainder if accumulated across epochs. We choose enough updates to make a useful small-corpus continuation experiment. Gradient noise and optimization behavior therefore differ materially.\n\n**D05 - Physical microbatch 1, accumulation, and epoch remainders.** One sequence fits the A100 with our numerical state. FP32 gradient buffers reduce repeated BF16 accumulation rounding. Averaging the final three-sequence window by its actual size preserves an example-average update, but creates smaller batches at epoch boundaries. Microbatch-dependent behavior and finite-precision reductions prevent exact equivalence to a large physical batch.\n\n**D06 - Scaled learning-rate trajectory and reset optimizer.** The checkpoint does not include the historical optimizer or scheduler state. We initialize new moments and use the short schedule in section 4. This avoids a warmup longer than the entire experiment and an unsupported assumption that a from-scratch rate is suitable for continuation. It also makes direct replication of the original trajectory impossible.\n\n**D07 - CPU offloading and precision choices.** CPU FP32 masters/moments and GPU FP32 accumulation are explicit, tested choices. They avoid model/optimizer quantization and preserve small updates. CPU transfers and stock CPU kernels increase time. They do not establish the original optimizer precision or reproduce its device/reduction order.\n\n**D08 - Reference-code epsilon.** The paper's idealized masking description and G's numerical implementation differ at the lower endpoint. We choose the executable reference formula and record that choice. Very-low-mask events are therefore slightly different from an exact unrestricted uniform rate.\n\n**D09 - Document boundaries, order, and tail.** We insert one EOS, omit an added BOS, concatenate in frozen source order, then shuffle blocks. We retain the final 1637-token block without padding so no original text is silently discarded or synthetic targets learned. The exact historical separator, order, shuffle/repacking, and corpus-tail policies are unavailable. Our extra short tail is one block per epoch in addition to stochastic cropping.\n\n**D10 - Per-occurrence random crop at microbatch 1.** A cropped full block uses an integer length from 1 to 4096; the already-short final block is cropped only within its available length. The released example applies a common crop to a physical batch; our virtual batch may contain different lengths. We normalize each example by its actual length and average examples. The resulting length/sample weighting may differ from the distributed implementation.\n\n**D11 - Declared optimizer defaults and grouping.** Betas, epsilon, clipping, and decay on all parameters are not asserted as historical settings. These assumptions fix otherwise ambiguous implementation choices. They can affect learning, particularly norm/bias regularization and small gradients. The run records them so a later sensitivity experiment can change one factor at a time.\n\n**D12 - Different machine and software stack.** One A100, CPU offload, modern PyTorch/Transformers, and recomputation replace the original distributed environment. This changes throughput and floating-point reduction order. No claim of bitwise reproduction is made. The original architecture and objective are retained.\n\n**D13 - Diagnostic evaluation and model-only checkpointing.** The lightweight evaluation below is tailored to this study. Checkpoints support later extraction analysis but omit the large training state to respect Drive capacity. These choices affect observability and recovery, not the intended uninterrupted optimizer update rule.\n\n**D14 - Exact-overlap filtering only for controls.** The requested training membership remains intact; only contaminated controls are excluded from the diagnostic stream. This is a study-specific evaluation decision. It is insufficient to remove near-duplicate or historical pre-training contamination and must not be represented as a complete deduplication solution.\n\n## 7. Unknown original details and our handling\n\nThese are bounded claims about the reviewed public material, not proof that the information could never be recovered. A new author disclosure or training-code release could resolve them. We keep known divergences separate from missing information.\n\n| ID | Question mark | Explicit handling in this experiment |\n|---|---|---|\n| U01 | Exact original document identities, versions, and full filtering outputs | Freeze the BBC revision; do not claim corpus replication. |\n| U02 | Original article-level exposure, sampling/repetition rates, and exact domain schedule | Record our block visits and duplicates; do not infer historical exposure from an epoch count. |\n| U03 | Presence of older/reprinted BBC wording in historical training | Use the untouched Base reference and flag this limitation; dates alone are insufficient. |\n| U04 | Full original deduplication implementation and retained overlap | Perform explicit exact-overlap audit; report that near-duplicate audit remains outstanding. |\n| U05 | Exact document separators, BOS/EOS choices, and boundary reset policy | Use one EOS, no added BOS, and ordinary contiguous bidirectional blocks. |\n| U06 | Exact sample order, shuffle algorithm, repacking cadence, worker sharding, and final-tail handling | Freeze packing, log seed/permutation hashes, retain the short tail. |\n| U07 | Adam beta coefficients | Set (0.9, 0.999) as an implementation assumption. |\n| U08 | Adam epsilon and related numerical variants | Set 1e-8; use stock torch AdamW. |\n| U09 | Which parameter groups receive weight decay | Apply 0.1 to every trainable parameter. |\n| U10 | Gradient-clipping policy and threshold | Use global FP32 norm clipping at 1.0. |\n| U11 | Training model/master-weight, gradient, and moment dtypes | Declare our BF16 compute and FP32 accumulation/masters/moments explicitly. Inference dtype in the paper does not resolve training-state precision. |\n| U12 | Loss scaling, mixed-precision underflow handling, stochastic rounding, and optimizer fusion | Use no GradScaler for BF16, no stochastic rounding, no fused/foreach CPU optimizer; abort nonfinite updates. |\n| U13 | Original loss-reduction details for variable lengths, per-rank averaging, and zero-mask edge cases | Follow G's normalization per sequence; average the actual virtual batch; keep zero-mask contributions zero. |\n| U14 | Correlation/stratification of mask rates across ranks or batches | Independent fresh per-example draws from the declared stream. |\n| U15 | Training seeds, full RNG states, deterministic-kernel settings | Seed our experiment, record versions and hashes; deterministic CUDA equivalence is not claimed. |\n| U16 | Exact distributed parallelism, accumulation, gradient-reduction order, kernels, and activation-recomputation placement | Use one-GPU non-reentrant whole-layer checkpointing and FP32 buffers; document hardware and versions. |\n| U17 | Original optimizer/scheduler state corresponding to the released weights | Initialize a new optimizer; label this continuation rather than exact resumption. |\n| U18 | Every operational restart, data replay, skipped update, recovery choice, and full intermediate training log | Record our own events/failures; do not reconstruct an unavailable historical trajectory. |\n| U19 | Whether the public reference snippet exactly matches every original production-training detail | Treat G as behavioral guidance, not a complete original training-system release. |\n| U20 | A published, validated recipe for continuing Base on this particular small BBC corpus | Our batch, schedule, and three-epoch budget are declared experimental choices; no quality guarantee is inferred. |\n\nThe checkpoint's architecture and tokenizer configuration are available and preserved; these are not counted as unknown merely because the original training framework is missing. Original pre-training and SFT settings are not interchangeable.\n\n## 8. Diagnostics and later memorization analysis\n\nBefore training and after each epoch, measure the same masked-diffusion objective on eight evenly spaced full blocks from the training stream and eight from the clean odd-control stream. Use four fixed mask draws per block and matched seeds across groups. Evaluation uses no gradient, model evaluation mode, and isolated RNG state so it does not alter subsequent training noise.\n\nThese are descriptive fixed-noise training-objective diagnostics. They are not an exact log-likelihood, an extraction score, a calibrated membership-inference attack, or a complete memorization test. The small number of mask draws is chosen for economical monitoring and is not asserted to reproduce the paper's likelihood-evaluation procedure. Controls never affect gradients, schedule, stopping, or model selection in this fixed run.\n\nThe following later analysis choices remain to be specified before making scientific memorization claims: prefix/suffix lengths, extraction sample selection, decoding/remasking settings and seeds, generated length, exact-match and near-match criteria, duplicate/boilerplate handling, statistical aggregation, and uncertainty estimates. Save all three epoch models so that this analysis can compare exposure levels against the same untouched base reference. Do not report diagnostic loss improvements as verbatim memorization rates.\n\n## 9. Model sizes and Drive retention\n\nEach of the 8,015,581,184 parameters requires two bytes in the saved BF16 model. Tensor payload is exactly **16,031,162,368 bytes = 16.031 GB = 14.930 GiB**. The existing base shards occupy 16,031,197,112 bytes, excluding tokenizer/configuration/custom-code files; this independently verifies the payload calculation. Fine-tuned snapshots retain the same shapes and dtype, so they have approximately the same size. Shard headers, metadata, tokenizer, and custom code add a small overhead.\n\n| Retained object | Approximate decimal size | Approximate binary size |\n|---|---:|---:|\n| One BF16 model | 16.03 GB | 14.93 GiB |\n| Epochs 1, 2, and 3 model snapshots together | 48.09 GB | 44.79 GiB |\n| Original base cache plus all three snapshots | 64.12 GB | 59.72 GiB |\n| One CPU FP32 master copy | 32.06 GB | 29.86 GiB |\n| Two FP32 Adam moment tensors together | 64.12 GB | 59.72 GiB |\n| Master copy plus optimizer moments | 96.19 GB | 89.58 GiB |\n| BF16 model plus full master/moment state | 112.22 GB | 104.51 GiB |\n\n**Retention policy:** keep the original cache and three separate epoch model-only snapshots. Save the tokenizer, configuration, needed custom model code, run/epoch metadata, and shard hashes. Do not write FP32 master weights or optimizer moments to Drive. Log observed sizes after each save.\n\nModel-only checkpoints are suitable for loading the BF16 model for inference and later analysis. They do not permit exact training continuation: moments, sub-BF16 master-weight information, accumulation buffers, and full RNG/scheduler state are absent. An interruption may require restarting this fixed experiment from Base; loading an intermediate model would be a different continuation run and must receive a new run identifier.\n\nOn the inspected mount, about 102 GiB was available before writing these snapshots. Approximately 45 GiB for the three model snapshots fits that reported capacity; retaining resumable FP32 states alongside them does not. Recheck free space before training and each save, and do not delete unrelated Drive content. Reported mount capacity is an operational check, not an independent billing/quota guarantee. Other datasets, logs, and user files consume additional space.\n\nCheckpoint directories are considered valid only after all shards and supporting files have been written, their headers/shapes/dtypes checked, hashes recorded, and a `COMPLETE.json` marker produced. A partially written directory is not a completed checkpoint. All large writes occur on Google Drive from Colab; no weights or dataset are downloaded to the Windows workspace.\n\n## 10. Feasibility evidence and runtime estimate\n\nThe selected numerical setup completed one warmup and two timed optimizer updates, each accumulating 16 real 4096-token blocks. All 48 sample losses and update gradient norms were finite. Master parameters changed. A separate small-tensor test compared the gradient-accumulation mechanism to full-batch autograd across two windows.\n\n| Quantity | Measured result |\n|---|---:|\n| Trainable parameters | 8,015,581,184 |\n| Mean steady-state update | 72.29 seconds, including all 16 microbatches and CPU transfers |\n| Throughput | About 907 tokens/second |\n| Peak allocated GPU memory | 52.7 GiB |\n| Peak reserved GPU memory | 71.3 GiB |\n| Peak process CPU RAM | 125.0 GiB |\n| Steady-state three-epoch projection | About 3 hours 38 minutes |\n\nThis projection excludes setup, data preparation, diagnostics, model saving, and interruptions. Smaller final accumulation windows add some optimizer overhead relative to the steady-state token-rate projection. Random short examples slightly change the compute mix. Budget roughly 4-5 hours plus evaluation/saving variability; this is a forecast, not a completed-run measurement or guarantee. The production status/events will provide actual times.\n\nEarlier 1024/4096 BF16-moment and quantized-optimizer benchmarks measured different numerical setups. Their faster estimates do not describe this FP32-offloaded run. The original benchmark state is discarded; production starts from the original frozen model.\n\n## 11. Execution records, validation, and failure handling\n\nThe production run records model/dataset revisions, environment versions, source and configuration hashes, method-document hash, block-permutation hashes, selected counts, actual processed tokens, random-crop counts, loss, gradient norm, learning rate, update/epoch number, timings, diagnostic results, and checkpoint manifests. Sources, configuration, and documentation are copied into the run directory before any optimizer update.\n\nImplementation checks cover the scheduler's exact endpoints, averaged gradients for a full and a remainder accumulation window, zero-mask loss behavior, independent FP32 masters, and checkpoint indexing/header validation. Data preparation must reproduce the frozen selected-token count. Any mismatch is an error rather than silently changing membership or data volume.\n\nThe trainer aborts on nonfinite losses/gradients, missing or corrupt inputs, mismatched parameter counts, insufficient reported save space, or checkpoint-validation failure. Errors and tracebacks are written into the run status. Existing completed run directories are not overwritten. No automatic restart from a model-only snapshot is allowed under the same run identifier.\n\n## 12. Artifact map and reproducibility\n\nAll large artifacts live under `/content/drive/MyDrive/LLaDA-ft/`:\n\n- `hf-cache/`: original pinned model, tokenizer, and dataset caches.\n- `datasets/manifest.json`: original pinned date/parity selection metadata.\n- `datasets/bbc_even_after_2025_02.jsonl`: frozen selected article membership and text.\n- `datasets/train_tokens.npy`: complete selected token stream.\n- `datasets/bbc_odd_controls.jsonl`: odd records and exact-overlap flags.\n- `datasets/control_tokens_clean.npy`: diagnostic control token stream.\n- `datasets/training_data_manifest.json`: counts, duplicate audit, token-file hashes.\n- `docs/LLADA_BBC_METHOD.md`: this protocol; the run also captures its own immutable copy.\n- `code/`: preparation, training, validation, and benchmark source.\n- `runs/bbc_fp32_3ep_v1/`: frozen configuration/sources/methods, status, events, and diagnostics.\n- `checkpoints/bbc_fp32_3ep_v1/epoch-001/`, `epoch-002/`, `epoch-003/`: completed model-only snapshots.\n- `logs/`: preparation, benchmark, and production process logs.\n\nThe Windows workspace contains only small code, configuration, documentation, and scalar-result files. The existing runtime is reused; no new runtime is provisioned. A reader reproducing the experiment must obtain the pinned original model and dataset, create the recorded environment, run preparation/validation, and start the production trainer with the frozen configuration. Exact CUDA bit patterns are not promised even with the same seed.\n\nAny later change to membership, numerical precision, objective, batching, schedule, exposure budget, or checkpoint-resume behavior requires an updated protocol and a new run identifier. New evidence about the authors' procedure should be entered as a resolved unknown, rather than retroactively describing an assumption as a confirmed match.\n\n\n## 13. Version 1.1 amendment: interrupted run, verified API exports, and replacement run\n\nThis section overrides the storage/execution/retention statements above where they differ. Numerical training, membership, objective, effective batch, seeds, learning-rate schedule and three-epoch budget remain unchanged. The v1 run directory and original protocol are retained as historical records, not overwritten. The replacement configuration is `llada_training_config_v2.json`.\n\n### Incident established from persisted evidence\n\nBoth the mounted files and direct Drive API returned the v1 status at epoch 1, update 54 of 61. The last persisted update was 21 October 2026 at 21:14:09 UTC (23:14:09 Europe/Zurich). No epoch-completion, checkpoint-start or checkpoint-completion event was present, and no model checkpoint existed in Drive. The runtime was replaced before this audit. This evidence does not demonstrate a large-file write failure: there is no persisted checkpoint attempt. The precise shutdown cause and any unpersisted activity cannot be reconstructed. In-memory weights and optimizer state from the old runtime are unavailable, so the new run starts from the original Base model, not from update 54.\n\n### Observed large-file transport test\n\nA real 2,948,598,128-byte base-model shard was copied to local Colab disk and uploaded in 64 MiB resumable chunks through authenticated Drive API v3. The returned file size, server MD5 and server SHA256 all matched the local file; transfer/verification took about 85.6 seconds after the local copy. Only the new temporary remote test copy was deleted afterwards. The original model cache was untouched. A verification receipt is saved under `LLaDA-ft/storage-validation/resumable-upload-20261022/`. This was a transport test, not a trained model.\n\n### Replacement checkpoint path\n\nThe trainer first saves a complete original Base-model transport-preflight checkpoint before any optimizer update. It validates the local safetensors headers/index/shapes/dtypes and hashes, then uploads and verifies all files through the API. Production training begins only after this full-size checkpoint path succeeds. The preflight artifact is labelled `base-transport-preflight` and must never be represented as a fine-tuned model.\n\nEvery model export is serialized on the runtime-local `/content/llada-checkpoint-staging/` filesystem rather than on the Drive FUSE mount. The authenticated API uploads shards and sidecars using resumable `MediaFileUpload` requests with SDK retry handling. For each file it verifies server size and MD5, and server SHA256 when supplied. Local SHA256 values are recorded. The remote `COMPLETE.json` marker is sent last. A run-journal upload receipt records file IDs and verified checksums. No success is declared merely because a file appeared on the mount.\n\nTraining additionally exports a periodic model snapshot every 20 optimizer updates, except when no more than five updates remain in the epoch (the epoch checkpoint then follows shortly). At each successful new periodic upload the previous periodic snapshot created by this run is removed. Before removal, its child IDs must exactly match the upload receipt; unexpected contents stop automatic cleanup. An epoch export also supersedes the previous periodic snapshot. All three epoch snapshots are retained. Uploads pause optimization and do not change model weights or intended gradient calculations.\n\nThese remain model-only checkpoints: FP32 masters, optimizer moments, full gradient/RNG/scheduler recovery state are not saved. A periodic model allows recovery of a trained model for analysis, but does not enable exact optimizer resumption. Continuing from it would be a separately labelled continuation with reset optimizer state. No automatic exact-resume claim is made.\n\n### Execution and status\n\nThe actual training subprocess is executed as blocking work in the Colab notebook kernel through `launch_training_in_kernel.py`, rather than relying only on an SSH-launched background process while the kernel is idle. This runs real work, not a synthetic keepalive. Colab can still stop or expire a runtime; indefinite runtime survival is not guaranteed.\n\nSmall mount-based records remain in `runs/bbc_fp32_3ep_v2_api/`. Independent API-owned copies and an overwritable API-owned `status_api.json` are under `run-api-records/bbc_fp32_3ep_v2_api/`. Model folders are under `checkpoints/bbc_fp32_3ep_v2_api/`. The API journal is authoritative for remote save verification; a stale status marked running does not prove that its old process remains alive. Credentials and resumable-session URLs are never stored in checkpoint metadata or printed.\n\n### Resource and interpretation changes\n\nResumable API transfers add time. The measured 2.95 GB transport rate suggests roughly eight minutes per 16 GB export, plus local serialization, hashing and variable network time. The initial full-size preflight, up to six periodic exports and three epoch exports can add approximately an hour or more to the earlier compute-only forecast. Actual per-export durations are logged; previous 4-5 hour estimates should not be treated as an upper bound for this more durable workflow.\n\nDrive API quota, not only the mount filesystem report, is checked. The peak-space check budgets the three retained epoch snapshots plus two periodic/staging copies and headroom, although obsolete periodic copies are collected only after a new verified copy exists. The local disk must also have room for a complete model. Upload failures abort training with a recorded error instead of silently advancing without a verified save. No unrelated user files or old run artifacts are deleted.\n\nValidation covers numerical training checks from v1, completion-marker ordering, checksum mismatch handling, immutable-file conflicts, owned-periodic cleanup, and the real large-shard upload. The first full-model preflight is a production gate that tests the entire export path before another long training run.\n\n\n## 14. Version 1.2 amendment: local pinned input weights\n\nThe v2 initialization failed before any optimizer update because the persisted Drive cache lacked `model-00001-of-00006.safetensors`. The failure was recorded; no v2 trained model exists. This is a separate input-cache fault and is not evidence that the original v1 run attempted a model export. The precise reason the large cache entry was not durable is unconfirmed.\n\nThe replacement run is `bbc_fp32_3ep_v3_api`, configured by `llada_training_config_v3.json`. Download the same model revision directly from Hugging Face into the runtime-local `/content/llada-base-model` directory and load it from there. The configuration, tokenizer and Python model files come from that pinned snapshot too. The six weight-shard sizes must sum to 16,031,197,112 bytes before startup. Runtime-local files are disposable caches, not durable experiment outputs; no large file is copied to the Windows workspace.\n\nAll v1/v2 failed-run records remain untouched. The checkpoint and API-journal paths in section 13 use the v3 run identifier for this run. The v3 immutable run record includes this complete updated protocol. The numerical method is unchanged; only the input-cache location and recovery/execution/storage mechanisms have changed. A full original-model serialization and verified API upload remains a gate before any new training update.\n\n\n## Standalone notebook execution note\n\nThe notebook generates a unique run name and embeds all implementation files. The shared path module is reduced to paths/environment only; Hugging Face caches and the pinned Base weights reside on Colab local disk. Data reuse requires canonical SHA-256 token-file checks; missing data is rebuilt from the pinned revisions and original global index rule. A genuine Drive API write/read check and resource/quota checks precede large downloads. The numerical training, diagnostics, checkpoint validation and upload implementation remain the protocol above. Python dependencies are pinned where tested; Colab supplies its installed PyTorch/CUDA version, which is constrained against replacement and recorded per run. Model-only snapshots do not restore FP32 optimizer state, data order or RNG state.\n\nThe notebook quota calculation supersedes the earlier conservative storage budget: with three retained epochs, peak simultaneous Drive storage is two completed epochs plus one old rolling snapshot plus one replacement upload (four model payloads). Colab-local serialization does not consume Drive quota. The guard requires four payloads plus 2 GiB headroom, about 66.3 GB free at startup. This covers this run only; concurrent uploads by other jobs must be budgeted separately.\n"
methods_path = ROOT / 'docs' / CONFIG['methods_filename']
methods_path.write_text(METHODS_TEXT, encoding='utf-8')
print('METHODS_SAVED:', methods_path)

## 10. Run the focused checks
CPU checks cover masked loss alignment, zero-mask handling, effective batch averaging including the short final batch, FP32 master/state behavior, learning-rate endpoints, checkpoint validation, upload failure gates and completion-marker ordering. No model training occurs here.

In [ ]:
for filename in ['validate_training_setup.py', 'validate_drive_checkpoint_upload.py']:
    subprocess.run([PYTHON, '-u', '/content/' + filename], check=True)

## 11. Train — this cell runs for hours

Run once. Leave this blocking cell running. It first evaluates baseline diagnostics and uploads an **untrained** full-model transport test. Training starts only after that upload has passed. It then saves rolling snapshots every 20 global updates when more than five updates remain in the epoch, plus all three epoch snapshots. Each new successful snapshot replaces the preceding rolling snapshot; completed epoch snapshots remain.

Progress is printed live and logged under `My Drive/LLaDA-ft/logs/`. A checkpoint is usable only when the Drive API receipt is verified and its cloud folder contains `COMPLETE.json`. An upload failure stops the run. A Colab termination can still lose progress since the most recent completed upload. No automatic exact resume is claimed.

In [ ]:
assert CONFIG_PATH.exists()
assert json.loads(CONFIG_PATH.read_text())['run_name'] == RUN_NAME
assert not (ROOT / 'runs' / RUN_NAME).exists(), 'This run already started; inspect its status instead of restarting.'
LOG_PATH = ROOT / 'logs' / (RUN_NAME + '.log')
print('Starting', RUN_NAME, '\nLog:', LOG_PATH, flush=True)
with LOG_PATH.open('a', encoding='utf-8', buffering=1) as log:
    process = subprocess.Popen([PYTHON, '-u', '/content/train_llada_bbc.py', '--config', str(CONFIG_PATH)],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            log.write(line)
            print(line, end='', flush=True)
        returncode = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
if returncode:
    raise RuntimeError(f'Training exited with code {returncode}. Inspect the log and API status; do not assume completion.')
print('Training process finished. Run the final cloud verification cell below.', flush=True)

## 12. Verify all three epoch checkpoints in Drive
This checks the API journal and every epoch receipt against current cloud file sizes/checksums, including the completion marker. A success message is printed only after all three checks pass. This cell runs after training; the same notebook kernel cannot run another cell while the training cell is busy.

In [ ]:
%%writefile /content/notebook_verify_final.py
import json
from pathlib import Path
from drive_checkpoint_upload import DriveCheckpointStore
config=json.loads(Path('/content/llada_training_config.json').read_text())
store=DriveCheckpointStore()
parent='root'
for part in ['LLaDA-ft','run-api-records',config['run_name']]:
    parent=store.folder(parent,part,create=False)
    assert parent, 'Run API journal missing.'
def read_json(name):
    files=store.children(parent,name)
    assert len(files)==1, 'Missing or ambiguous journal file: '+name
    return json.loads(store.service.files().get_media(fileId=files[0]['id']).execute(num_retries=5))
status=read_json('status_api.json')
print('RUN_STATUS',json.dumps(status,indent=2))
assert status['status']=='complete', 'Run did not report successful completion.'
for epoch in [1,2,3]:
    receipt=read_json(f'epoch-{epoch:03d}_upload_receipt.json')
    assert receipt['verified'] and receipt['epoch']==epoch and 'COMPLETE.json' in receipt['files']
    for name,expected in receipt['files'].items():
        actual=store.service.files().get(fileId=expected['id'],fields='id,name,size,md5Checksum,sha256Checksum').execute(num_retries=5)
        assert actual['name']==name and actual['size']==expected['size']
        assert actual['md5Checksum']==expected['md5Checksum']
        if actual.get('sha256Checksum'):
            assert actual['sha256Checksum']==expected['local_sha256']
    marker=receipt['files']['COMPLETE.json']
    complete=json.loads(store.service.files().get_media(fileId=marker['id']).execute(num_retries=5))
    assert complete['epoch']==epoch and complete['run']==config['run_name']
    print(f'EPOCH_{epoch}_CLOUD_VERIFIED',receipt['folder_url'],flush=True)
print('ALL_THREE_TRAINED_EPOCH_MODELS_VERIFIED_IN_DRIVE')

In [ ]:
subprocess.run([PYTHON, '-u', '/content/notebook_verify_final.py'], check=True)